### Import thư viện

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.preprocessing import StandardScaler
from scipy.stats import uniform, randint
import numpy as np
from sklearn.model_selection import KFold, cross_val_score

import warnings
from sklearn.exceptions import ConvergenceWarning

In [ ]:
warnings.filterwarnings("ignore", category=ConvergenceWarning)

## Đọc file

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Đồ án/Gold_data.csv')
df

,ordinal_col__Accommodation_Type,ordinal_col__Room_Level,onehot_col__Room/Bed Quantity_01 giường,onehot_col__Room/Bed Quantity_02 giường,onehot_col__Room/Bed Quantity_1 giường,onehot_col__Room/Bed Quantity_1 giường đôi,onehot_col__Room/Bed Quantity_1 giường đơn,onehot_col__Room/Bed Quantity_1 phòng,onehot_col__Room/Bed Quantity_10 giường,onehot_col__Room/Bed Quantity_12 giường,...,beachs,destination,Adults,Children,count_nearest_airport,distance_to_nearest_airport,day,month,is_weekend,day_range
0,7.0,6.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,5.0,9.4,1.0,0.0,2.0,9.0,15.0,5.0,0.0,15.0
1,7.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,5.0,8.7,1.0,0.0,2.0,6.0,15.0,5.0,0.0,15.0
2,7.0,6.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,5.0,8.7,1.0,0.0,2.0,5.0,15.0,5.0,0.0,15.0
3,7.0,4.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,5.0,9.2,1.0,0.0,2.0,6.0,15.0,5.0,0.0,15.0
4,8.0,3.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,5.0,8.9,1.0,0.0,2.0,9.0,15.0,5.0,0.0,15.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242235,6.0,3.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,1.0,8.1,4.0,2.0,1.0,7.0,14.0,8.0,0.0,93.0
242236,7.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,8.7,4.0,2.0,1.0,0.0,14.0,8.0,0.0,93.0
242237,6.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,10.0,4.0,2.0,1.0,3.0,14.0,8.0,0.0,93.0
242238,7.0,3.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,9.2,4.0,2.0,1.0,5.0,14.0,8.0,0.0,93.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 242240 entries, 0 to 242239
Columns: 154 entries, ordinal_col__Accommodation_Type to day_range
dtypes: float64(154)
memory usage: 284.6 MB


## Chia dữ liệu

In [ ]:
target = 'Price'
X = df.drop(target, axis=1)
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Kích thước tập huấn luyện:', X_train.shape)
print('Kích thước tập kiểm thử:', X_test.shape)


Kích thước tập huấn luyện: (193792, 153)
Kích thước tập kiểm thử: (48448, 153)


In [ ]:
# Hàm tính R2 Adjust
def r2_adjusted(r2, n, p):
  r2_adjusted = 1 - (1 - r2) * (n - 1) / (n - p - 1)
  return r2_adjusted

In [ ]:
# Áp dụng StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Huấn luyện mô hình

### Đánh giá bằng kfold cross validation và chạy bằng mô hình với các siêu tham số mặc định

In [ ]:
# Cài đặt mô hình
model_lasso = Lasso()
# Thực hiện cross-validation để đánh giá dữ liệu train
scores_mae = -cross_val_score(model_lasso, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
scores_r2 = cross_val_score(model_lasso, X_train, y_train, cv=5, scoring='r2')

# # Tính R2 adjusted cho từng fold
# n_train = X_train.shape[0]
# p_train = X_train.shape[1]
# scores_r2_adj = [r2_adjusted(r2, n_train, p_train) for r2 in scores_r2]

# print('Result on train data:')
# print("   MAE trên mỗi fold:", scores_mae)
# print("   Mean MAE:", np.mean(scores_mae))
# print("   Std  MAE:", scores_mae.std())
# print("\n   R2 trên mỗi fold:", scores_r2)
# print("   Mean R2:", np.mean(scores_r2))
# print("   Std R2:", scores_r2.std())
# print("\n   R2 Adjusted trên mỗi fold:", scores_r2_adj)
# print("   Mean R2 Adjusted:", np.mean(scores_r2_adj))
# print("   Std R2 Adjusted:", np.std(scores_r2_adj))

print('Result on train data:\n')
print('   Mean MAE:', scores_mae.mean())
print('   MAE standard deviation:', scores_mae.std())
print('\n   Mean R2:', scores_r2.mean())
print('   R2 standard deviation:', scores_r2.std())
print('\n   Adjusted R2:', r2_adjusted(scores_r2.mean(), len(X_train), X_train.shape[1]))


Result on train data:

   Mean MAE: 539366.1548476844
   MAE standard deviation: 2559.1297800346138

   Mean R2: 0.549116491717465
   R2 standard deviation: 0.005493195986288736

   Adjusted R2: 0.5487602332518373


In [ ]:
model_lasso.fit(X_train, y_train)
y_pred = model_lasso.predict(X_test)

r2_score_default_model = r2_score(y_test, y_pred)
mae_score = mean_absolute_error(y_test, y_pred)

print('Result on test data:')
print('   R2 score:', r2_score_default_model)
print('   MAE score:', mae_score)
print('   R2 adjusted score:', r2_adjusted(r2_score_default_model, len(X_test), X_test.shape[1]))

Result on test data:
   R2 score: 0.5546746648285501
   MAE score: 537773.9178824526
   R2 adjusted score: 0.553263831675752


### Dùng GridSearch để tìm ra bộ siêu tham số tối ưu

In [ ]:
param_grid = {
    'alpha': [0.1, 0.5, 1.0],
    'max_iter': [500, 1000, 2000, 5000],
    'tol': [1e-4, 1e-3, 1e-2]
}

# Tạo grid search
grid_search = GridSearchCV(
    estimator=Lasso(),
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=5,
    n_jobs=-1
)

# Fit trên tập train
grid_search.fit(X_train, y_train)

# In ra bộ siêu tham số tốt nhất và điểm số tương ứng
print("Bộ siêu tham số tốt nhất:", grid_search.best_params_)
print("Điểm MAE tốt nhất từ GridSearch:", -grid_search.best_score_)

# Đánh giá mô hình tốt nhất trên tập kiểm thử
best_lasso_model = grid_search.best_estimator_
y_pred_best = best_lasso_model.predict(X_test)

r2_score_best_model = r2_score(y_test, y_pred_best)
mae_score_best_model = mean_absolute_error(y_test, y_pred_best)

print('\nResult tunned on test data:')
print('   R2 score của mô hình tốt nhất:', r2_score_best_model)
print('   MAE score của mô hình tốt nhất:', mae_score_best_model)
print('   R2 adjusted score của mô hình tốt nhất:', r2_adjusted(r2_score_best_model, len(X_test), X_test.shape[1]))

Bộ siêu tham số tốt nhất: {'alpha': 0.1, 'max_iter': 1000, 'tol': 0.0001}
Điểm MAE tốt nhất từ GridSearch: 539370.0042542752

Result tunned on test data:
   R2 score của mô hình tốt nhất: 0.5546779096475261
   MAE score của mô hình tốt nhất: 537773.9064814355
   R2 adjusted score của mô hình tốt nhất: 0.5532670867746241
